# AutoInsight Pipeline (Classic Flow)
In this classic workflow, the user manually selects the dataset and the machine learning method to apply. 
The AI is only involved at the very end to act as a frontend developer: it synthesizes the data profile 
and model execution metrics into a beautifully themed, custom HTML report based on your exact stylistic preferences.

In [10]:
import sys
import os
import json
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

# Ensure we can import from the root directory
sys.path.append(os.path.abspath('..'))

from autoinsight.src.clean import clean_data
from autoinsight.src.eda import generate_eda_profile
from shared.llm_client import OpenRouterClient
from shared.registry import list_methods, get_method

In [11]:
data_dir = '../data'
os.makedirs(data_dir, exist_ok=True)
csv_files = [f for f in os.listdir(data_dir) if f.endswith('.csv')]

if not csv_files:
    print('No CSV files found in ../data. Please add some. A dummy dataset will be used.')
else:
    dataset_dropdown = widgets.Dropdown(options=csv_files, description='Dataset:')
    display(dataset_dropdown)

Dropdown(description='Dataset:', options=('WA_Fn-UseC_-Telco-Customer-Churn.csv',), value='WA_Fn-UseC_-Telco-C…

In [12]:
if 'dataset_dropdown' in locals() and csv_files:
    selected_file = dataset_dropdown.value
    df = pd.read_csv(os.path.join(data_dir, selected_file))
    print(f'Loaded {selected_file} with shape {df.shape}')
else:
    # Fallback dummy dataset
    data = {
        'feature1': [1, 2, 1, 8, 9, 8, 2, 9],
        'feature2': [1.1, 1.9, 1.5, 8.1, 8.9, 9.2, 1.3, 8.5],
        'target': [0, 0, 0, 1, 1, 1, 0, 1]
    }
    df = pd.DataFrame(data)
df.head()

Loaded WA_Fn-UseC_-Telco-Customer-Churn.csv with shape (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [13]:
# 1. Clean Data & Generate EDA Profile
cleaned_df, cleaning_log = clean_data(df)
eda_profile = generate_eda_profile(cleaned_df)
print("EDA Summary:", eda_profile['summary'])

EDA Summary: {'num_rows': 7043, 'num_cols': 21}


In [14]:
# 2. Manual Method Selection
available_methods = list_methods()
if not available_methods:
    print('No methods found in registry. Please make sure they are implemented.')
else:
    method_dropdown = widgets.Dropdown(options=available_methods, description='Method:')
    display(method_dropdown)

No methods found in registry. Please make sure they are implemented.


In [ ]:
# 3. Execute Selected Method
if 'method_dropdown' in locals():
    selected_method = method_dropdown.value
    print(f'Executing {selected_method} manually...')
    
    func = get_method(selected_method)
    target_col = 'target' if 'target' in df.columns else None
    
    X = df.drop(columns=[target_col]) if target_col else df
    y = df[target_col] if target_col else None
    
    try:
        results = func(X, y)
        print('Execution Metrics:', results.get('metrics', {}))
    except Exception as e:
        print(f'Error executing method: {e}')

In [ ]:
# 4. Define HTML Theme / Tone for the LLM
theme_input = widgets.Text(
    value='Dark mode cyberpunk with glowing neon elements',
    placeholder='Enter visual theme and tone',
    description='HTML Theme:',
    layout=widgets.Layout(width='80%')
)
display(theme_input)

In [ ]:
# 5. Request Custom HTML Report from LLM
client = OpenRouterClient()
theme = theme_input.value if 'theme_input' in locals() else 'Clean and corporate'

prompt = f"""
I have profiled a dataset and manually executed a machine learning model on it.
EDA Profile: {json.dumps(eda_profile)}
Model Executed: {selected_method}
Execution Results: {json.dumps(results.get('metrics', {}))}

Your task is to act as an expert data scientist and elite frontend developer. 
Generate a complete, standalone HTML file that acts as the final report. 
The report must include a well-written narrative summary explaining the EDA findings and the model execution metrics. 

CRITICAL REQUIREMENT: The visual theme, styling, and tone of the HTML report MUST strictly follow this prompt: '{theme}'.
Make it visually stunning using entirely embedded CSS (no external CSS files, though CDNs for fonts are fine). 
Return ONLY the raw HTML code, starting with <!DOCTYPE html> and ending with </html>. Do not wrap it in markdown block quotes.
"""

print(f"Generating HTML report matching theme: '{theme}'...")
html_response = client.generate(prompt)

# Clean up any markdown formatting the LLM might have erroneously added
if html_response.startswith('```html'):
    html_response = html_response[7:].strip()
elif html_response.startswith('```'):
    html_response = html_response[3:].strip()
if html_response.endswith('```'):
    html_response = html_response[:-3].strip()

output_path = 'reports/autoinsight_custom_report.html'
os.makedirs(os.path.dirname(output_path), exist_ok=True)
with open(output_path, 'w', encoding='utf-8') as f:
    f.write(html_response)

print(f"Custom HTML Report successfully generated at {os.path.abspath(output_path)}")